# Mistral-Nemo server for the Ingaze engine

1. Menu **Runtime → Change runtime type → T4 GPU → Save**.
2. Menu **Runtime → Run all**.
3. Wait about 5 minutes. The last cell prints a line like `LOCAL_LLM_BASE_URL=https://xxxx.trycloudflare.com/v1`. Copy it into `user\.env` on your PC.
4. Keep this tab open while you use the app.

In [ ]:
# 1) Check that a GPU is attached (should show a Tesla T4)
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2) Install Ollama (the model server) and cloudflared (the tunnel)
!apt-get -qq install -y zstd pciutils > /dev/null
!curl -fsSL https://ollama.com/install.sh | sh
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared && chmod +x cloudflared

In [ ]:
# 3) Start the Ollama server in the background
import subprocess, os, time, re
env = dict(os.environ, OLLAMA_HOST="0.0.0.0:11434", OLLAMA_NUM_PARALLEL="4", OLLAMA_KEEP_ALIVE="-1")
subprocess.Popen(["ollama", "serve"], env=env, stdout=open("ollama.log", "w"), stderr=subprocess.STDOUT)
time.sleep(5)
print(open("ollama.log").read()[-500:])

In [ ]:
# 4) Download Mistral-Nemo (~7 GB, takes a few minutes)
!ollama pull mistral-nemo

In [ ]:
# 5) Quick test: the model answers on the GPU
!curl -s http://localhost:11434/v1/chat/completions -H 'Content-Type: application/json' -d '{"model":"mistral-nemo","messages":[{"role":"user","content":"Say hi in 3 words"}]}'

In [ ]:
# 6) Open a public tunnel and print the line to paste into user\.env
subprocess.Popen(["./cloudflared", "tunnel", "--url", "http://localhost:11434"],
                 stdout=open("cf.log", "w"), stderr=subprocess.STDOUT)
url = None
for _ in range(30):
    time.sleep(2)
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", open("cf.log").read())
    if m:
        url = m.group(0); break
print("COPY THIS LINE INTO user\\.env:\n")
print(f"LOCAL_LLM_BASE_URL={url}/v1" if url else "Tunnel failed, run this cell again")